# 📊 10.15 Aggregations

Welcome to the final lesson of **SECTION D: DATAFRAMES (CORE INDUSTRY SKILL)**!

In the previous lessons, we loaded our data, cleaned up messy text, formatted dates, extracted nested JSON, and patched up our missing Null values. 

Our data is finally clean. Now, it is time to do what Data Engineers and Analysts actually get paid to do: **Calculate Business Insights**.

In this lesson, we will learn how to crush millions of rows of data down into meaningful summary metrics (like finding the total revenue per department, or the average salary by state) using Spark's Aggregation functions.

### 🎯 Learning Objectives
* Understand the "Split-Apply-Combine" logic of `groupBy()`.
* Master basic aggregations like `count()`, `sum()`, `avg()`, `min()`, and `max()`.
* Learn how to calculate multiple metrics at the exact same time using `agg()`.
* Understand the performance implications (Network Shuffles) of aggregating data.
* Compare how different data roles utilize aggregations.

In [ ]:
# 0. Setup: Let's start our SparkSession and create a clean corporate dataset.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Aggregations").master("local[*]").getOrCreate()

data = [
    ("Alice", "Sales", 60000.0, 150000.0),
    ("Bob", "Sales", 55000.0, 120000.0),
    ("Charlie", "Engineering", 90000.0, 0.0),
    ("Diana", "Engineering", 95000.0, 0.0),
    ("Eve", "Marketing", 70000.0, 50000.0),
    ("Frank", "Sales", 50000.0, 90000.0)
]
columns = ["emp_name", "department", "salary", "sales_revenue"]

df = spark.createDataFrame(data, columns)
print("Clean Corporate Data:")
df.show()

## 1. The Basics: `groupBy()`

When you want to summarize data, you rarely want to summarize the *entire* table. Usually, you want to summarize it by a specific category (e.g., Total Revenue *by Department*).

To do this, we use `.groupBy("column_name")`.

Think of `groupBy` as creating physical buckets. If we `groupBy("department")`, Spark creates a "Sales" bucket, an "Engineering" bucket, and a "Marketing" bucket, and drops all the matching rows into those respective buckets.

Once the data is in buckets, we apply an **Aggregation Function** to crush the rows in the bucket down into a single number.

<img src="../assets/Module_10/10_15_01.png" alt="A visual showing multiple individual employee records flowing into three distinct buckets labeled Sales, Engineering, and Marketing. A machine at the bottom of the buckets crunches the records into a single summary number for each bucket." width="75%">

In [ ]:
print("--- Simple Aggregations ---")

# 1. COUNT: How many employees are in each department?
print("Headcount by Department:")
df_count = df.groupBy("department").count()
df_count.show()

# 2. SUM: What is the total sales revenue generated by each department?
print("Total Revenue by Department:")
df_sum = df.groupBy("department").sum("sales_revenue")
df_sum.show()

## 2. Advanced: The `agg()` Function

In the example above, we calculated the Headcount in one DataFrame, and the Total Revenue in another. 

In the real world, your boss will ask you for a single report that shows the Headcount, the Total Revenue, the Average Salary, the Minimum Salary, and the Maximum Salary, all in one table!

We can calculate multiple metrics simultaneously using the **`.agg()`** (aggregate) function. 

Inside `.agg()`, we pass our specific PySpark `F` functions (like `F.sum`, `F.avg`, `F.min`, `F.max`) and use `.alias()` to give the resulting columns clean, readable names.

In [ ]:
print("--- Multiple Aggregations at Once ---")

# We group by the department, then pass multiple mathematical functions into .agg()
summary_df = df.groupBy("department").agg(
    F.count("emp_name").alias("total_employees"),
    F.sum("sales_revenue").alias("total_department_revenue"),
    F.avg("salary").alias("average_salary"),
    F.min("salary").alias("lowest_salary"),
    F.max("salary").alias("highest_salary")
)

# Adding an orderBy to make the final report look professional
final_report = summary_df.orderBy(F.desc("total_department_revenue"))

final_report.show(truncate=False)

## 3. The Hidden Cost: The Network Shuffle

It is incredibly important to remember the architecture lesson (**10.5 DAG & Job Execution**) when doing aggregations.

A `.groupBy()` is a **Wide Transformation**. 

If you have 100 computers, and half the "Sales" employees are on Computer A and half are on Computer B, Spark cannot calculate the total sum immediately. It must physically transfer the data over the network so all "Sales" records end up on the exact same machine. 

This is called a **Network Shuffle**. Aggregations are one of the most expensive and time-consuming operations in Big Data because they require massive amounts of network traffic.

In [ ]:
# Let's prove that a Shuffle is happening by checking the partitions!
print(f"Partitions BEFORE groupBy: {df.rdd.getNumPartitions()}")

print(f"Partitions AFTER groupBy: {final_report.rdd.getNumPartitions()}")

print("\nNotice the jump to 200 partitions? That proves Spark executed a Shuffle!")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different roles approach summarizing data?

| Feature | 🏛️ Traditional DBA / Analyst | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Syntax** | Uses SQL: `SELECT dept, SUM(sales) FROM table GROUP BY dept;` | **Uses PySpark DataFrame API: `df.groupBy('dept').agg(F.sum('sales'))`** | Uses Pandas: `df.groupby('dept').agg({'sales': 'sum'})` |
| **Execution** | The database engine handles everything on a single server. | **Must design the DAG blueprint and monitor the Network Shuffle that results from the `.groupBy()`.** | Relies on the Data Engineer to pre-aggregate massive datasets so they fit into laptop RAM. |
| **Performance Focus** | Optimizing SQL execution plans. | **Dealing with 'Data Skew' (e.g., if one department has 5 million people and the others have 10, the cluster will choke during the shuffle).** | Feature Engineering (creating new predictive columns out of averages). |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Writes aggressive `.groupBy()` operations everywhere without realizing the network cost. They build pipelines that take 5 hours to run because they force the cluster to shuffle the data 10 different times.
2. **Mid-Level DE (Your Current Phase):** Consolidates code using `.agg()`. A Mid-Level DE knows that if you need 5 different metrics (sum, avg, min, max, count), you should put them all inside a *single* `.agg()` function so the cluster only has to perform *one* Network Shuffle instead of five!
3. **Senior DE:** Masters pre-aggregation techniques. If a dataset is too big to shuffle, a Senior DE will write custom logic to partially aggregate the data on the local nodes *before* the network shuffle occurs, saving massive amounts of time and money.

### 🛠️ Where Data Engineering Fits in Modern Organizations
The DataFrame output of this lesson (the `summary_df`) is exactly what a Data Engineer saves to a Cloud Data Warehouse (like Snowflake or Redshift). Then, Business Intelligence (BI) tools like Tableau or PowerBI connect to that summary table to draw the beautiful charts the CEO sees every morning.

## 🔑 Key Takeaways

- **`.groupBy(col)`:** Categorizes data into distinct "buckets" based on unique values in a column.
- **`.agg()`:** Allows you to perform multiple different mathematical calculations (sum, average, max) on the grouped data at the exact same time.
- **`F.alias()`:** Used inside `.agg()` to rename the ugly default column names (like `sum(sales_revenue)`) into clean business terms (like `total_revenue`).
- **The Shuffle Penalty:** Grouping data is a Wide Transformation. It forces Spark to send data across the network, which is the most expensive operation in Big Data.

## 📝 Knowledge Check Quiz

**Question 1:** You need to calculate the `sum` of revenue, the `avg` of age, and the `count` of employees for every country in a dataset. Which is the most efficient way to write this in PySpark?
A) Create three separate DataFrames, doing a separate `.groupBy("country")` on each one, and then joining them together.
B) Use `df.groupBy("country").agg(F.sum("revenue"), F.avg("age"), F.count("emp_id"))` to do it all in a single pass.
C) Download the data to a CSV and use Excel.
D) It is impossible to calculate more than one metric per `.groupBy()`.

**Question 2:** When you run a `.groupBy()` command in PySpark, what hidden operation must the cluster perform under the hood?
A) It deletes the Driver node.
B) A Network Shuffle (a Wide Transformation). The executors must physically send data to each other across the network so that matching categories end up on the same machine.
C) It converts the data to a Pandas DataFrame.
D) It bypasses Lazy Evaluation and forces an immediate disk write.

**Question 3:** What is the purpose of using `.alias()` after an aggregation like `F.sum("salary")`?
A) It encrypts the data for security.
B) It tells Spark to skip the calculation if the salary is zero.
C) It renames the resulting column to something clean and readable (e.g., `total_salary`) instead of the ugly default name generated by the system.
D) It connects the data to an external database.

---

*Answers: 1: B, 2: B, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
Congratulations! You have completed **Section D**. You are now capable of reading data, cleaning data, transforming data, and aggregating data.

But up until now, we have only been working with *one* table at a time. In the real world, user profiles live in one database, and their purchase history lives in another. 

In **SECTION E: COMBINING DATA**, we will learn how to stitch disparate datasets together across a massive cluster using **Joins**. See you in **10.16 Joins in Spark**!